# Mesh Simplification

In this notebook, I wish to explore the basics of Mesh Simplification. The work in this notebook builds on the preliminary work done in [mesh-simplification.md](../mesh-simplification.md). In that file, we defined our problem statement as follows.

*"We would like to implement a mesh simplification algorithm that maintains the original coordinates and positions of our vertices, such that when they are loaded to a scene they can share the same vertex buffer"*

Current problems with simplification algorithms include the fact that they create new vertices in an attempt to maintain the original shape and minimize the overall error. We found that using the algorithm "Simplification: Quadric Edge Collapse Decimation’ in MeshLab while setting the parameter `optimalposition` to be False helped achieve our intended result. However, the order of our indicies was still not maintained.

Here, we explore the basics of the PyMeshLab API and implement our algorithm using Python while applying some data cleaning steps to maintain our vertex order in the decimated mesh.

In [1]:
import pandas as pd
import numpy as np
import pymeshlab 

## Basic Implementation of Filters

We import a mesh using the `MeshSet()` class in PyMeshLab.

In [4]:
ms = pymeshlab.MeshSet()
# ms.load_new_mesh('../../../models/foot/human-foot.obj')

Now that we have the mesh in memory, we should be able to view basic attributes relating to it. Here, we see the list of vertices we have available

In [4]:
# Need to create a reference to the current mesh
m = ms.current_mesh()
v_matrix = m.vertex_matrix()

v_matrix

array([[ 0.059608,  0.383419, -0.047925],
       [-0.030599, -0.01601 , -0.020009],
       [ 0.052395,  0.402345,  0.102574],
       ...,
       [ 0.174218,  0.030867,  0.461069],
       [ 0.163158,  0.035477,  0.397872],
       [-0.056806,  0.012867,  0.570864]], shape=(800, 3))

We see the familiar (800,3) shaped array implying 800 vertices each with x,y,z coordinates. What about faces?

In [5]:
m = ms.current_mesh()
f_matrix = m.face_matrix()

f_matrix

array([[715,  31,  33],
       [715,  33, 712],
       [ 86,  16,  17],
       ...,
       [786,  95,  44],
       [647, 799,  95],
       [799, 651,  95]], shape=(1586, 3), dtype=int32)

1586 faces and 3 columns, implying that our data has been pre-converted to triangles. [We saw in earlier EDA](obj-eda.ipynb) that our original obj file had a mix of quads and triangles. It seems that loading to meshlab has automatically converted all quads to triangles.

We can even convert to dataframe.

In [6]:
vertex_df = pd.DataFrame(v_matrix, columns=["X", "Y", "Z"])
vertex_df.head()

,X,Y,Z
0,0.059608,0.383419,-0.047925
1,-0.030599,-0.016010,-0.020009
2,0.052395,0.402345,0.102574
3,0.049393,-0.015811,-0.022423
4,-0.061567,0.421855,-0.066727


In [7]:
face_df = pd.DataFrame(f_matrix, columns=["1", "2", "3"])
face_df.head()

,1,2,3
0,715,31,33
1,715,33,712
2,86,16,17
3,86,17,83
4,59,795,796


Let's apply a filter. In [mesh-decimation.md markdown file](../mesh-simplification.md) we discussed working with MeshLab through the GUI. In that document, we applied a Decimation with Quadric error, while maintaining the original mesh vertices. Let's do the same through the Python API.

In the [API](https://pymeshlab.readthedocs.io/en/latest/filter_list.html#meshing_decimation_quadric_edge_collapse) we see that this function is called as a method on our MeshSet(). Let's do this, using the default arguments, except for `optimalplacement` which we shall set to `False`. This argument controls the placement of our decimated vertices. In this case, we want our decimated vertices to maintain their original location coordinates.

In [8]:
ms.meshing_decimation_quadric_edge_collapse(optimalplacement=False)

Let's see what the result looks like. We're expecting a reduction in the number of vertices.

In [9]:
m = ms.current_mesh()
v_matrix_decimate = m.vertex_matrix()

v_matrix_decimate

array([[ 0.059608,  0.383419, -0.047925],
       [-0.030599, -0.01601 , -0.020009],
       [ 0.052395,  0.402345,  0.102574],
       ...,
       [ 0.100379, -0.002407,  0.430997],
       [ 0.130661,  0.005967,  0.426889],
       [ 0.174218,  0.030867,  0.461069]], shape=(403, 3))

Indeed we observe that our mesh has reduced vertices. Let's convert to dataframe and see what these vertices look like.

In [10]:
vertex_df_decimate = pd.DataFrame(v_matrix_decimate, columns = ["X", "Y", "Z"])
vertex_df_decimate.head()

,X,Y,Z
0,0.059608,0.383419,-0.047925
1,-0.030599,-0.016010,-0.020009
2,0.052395,0.402345,0.102574
3,0.049393,-0.015811,-0.022423
4,-0.061567,0.421855,-0.066727


In order to see whether our vertices in `vertex_df_decimate` match with the original, I will apply the following transformations.

1) Create a new column containing the tuple of (x,y,z) coordinates for each dataframe.
2) Merge dataframes based on this column (innerjoin).
3) Verfiy that the number of rows in each case is the same (i.e. no unmatched rows)

In [11]:
vertex_df['coords'] = vertex_df[["X", "Y", "Z"]].apply(tuple, axis=1)
vertex_df = vertex_df.reset_index()
vertex_df.head()

,index,X,Y,Z,coords
0,0,0.059608,0.383419,-0.047925,"(0.059608, 0.383419, -0.047925)"
1,1,-0.030599,-0.016010,-0.020009,"(-0.030599, -0.01601, -0.020009)"
2,2,0.052395,0.402345,0.102574,"(0.052395, 0.402345, 0.102574)"
3,3,0.049393,-0.015811,-0.022423,"(0.049393, -0.015811, -0.022423)"
4,4,-0.061567,0.421855,-0.066727,"(-0.061567, 0.421855, -0.066727)"


In [12]:
vertex_df_decimate['coords'] = vertex_df_decimate[["X", "Y", "Z"]].apply(tuple, axis=1)
vertex_df_decimate = vertex_df_decimate.reset_index()
vertex_df_decimate.head()

,index,X,Y,Z,coords
0,0,0.059608,0.383419,-0.047925,"(0.059608, 0.383419, -0.047925)"
1,1,-0.030599,-0.016010,-0.020009,"(-0.030599, -0.01601, -0.020009)"
2,2,0.052395,0.402345,0.102574,"(0.052395, 0.402345, 0.102574)"
3,3,0.049393,-0.015811,-0.022423,"(0.049393, -0.015811, -0.022423)"
4,4,-0.061567,0.421855,-0.066727,"(-0.061567, 0.421855, -0.066727)"


In [13]:
merged_df_vertex = pd.merge(vertex_df, vertex_df_decimate, how="outer", on="coords")
len(merged_df_vertex)

merged_df_vertex

,index_x,X_x,Y_x,Z_x,coords,index_y,X_y,Y_y,Z_y
0,645,-0.224832,0.030038,0.526462,"(-0.224832, 0.030038, 0.526462)",325.0,-0.224832,0.030038,0.526462
1,98,-0.223318,0.034579,0.441157,"(-0.223318, 0.034579, 0.441157)",92.0,-0.223318,0.034579,0.441157
2,299,-0.221680,0.043812,0.651764,"(-0.22168, 0.043812, 0.651764)",193.0,-0.221680,0.043812,0.651764
3,644,-0.221125,0.076088,0.533599,"(-0.221125, 0.076088, 0.533599)",324.0,-0.221125,0.076088,0.533599
4,99,-0.220750,0.085796,0.448143,"(-0.22075, 0.085796, 0.448143)",93.0,-0.220750,0.085796,0.448143
...,...,...,...,...,...,...,...,...,...
795,41,0.167509,0.009247,0.461653,"(0.167509, 0.009247, 0.461653)",38.0,0.167509,0.009247,0.461653
796,110,0.167673,0.057930,0.524913,"(0.167673, 0.05793, 0.524913)",102.0,0.167673,0.057930,0.524913
797,636,0.171731,0.034788,0.531401,"(0.171731, 0.034788, 0.531401)",317.0,0.171731,0.034788,0.531401
798,797,0.174218,0.030867,0.461069,"(0.174218, 0.030867, 0.461069)",402.0,0.174218,0.030867,0.461069


Perfect. We observe that each vertex in our 2 meshes are exactly aligned. To drive this point home, I shall repeat the above experiment with the same filter except this time I wont set the `optimalplacement` parameter to False. In this case, I expect that the model will create new vertices in our decimated mesh to maintain the overall shape. While this new mesh may look closer to the old one, it will likely create new vertices, implying that our total number of rows in the 

In [14]:
ms_2 = pymeshlab.MeshSet()
ms_2.load_new_mesh('../../../models/foot/human-foot.obj')

ms_2.meshing_decimation_quadric_edge_collapse(optimalplacement=True) #reverting this to True

m_test = ms_2.current_mesh()
v_matrix_test = m_test.vertex_matrix()

vertex_df_test = pd.DataFrame(v_matrix_test, columns = ["X", "Y", "Z"])

vertex_df_test['coords'] = vertex_df_test[["X", "Y", "Z"]].apply(tuple, axis=1)

test_df_vertex = pd.merge(vertex_df, vertex_df_test, how="outer", on="coords")
len(test_df_vertex)

975

As expected, we see an increase in the number of vertices, implying that we are creating new ones.

## Main Workflow

How do we maintain the same vertex index structure between our meshes? By default, this is not possible. Indices are assigned at generation. However, we know that the coordinates of our vertices are the same. We can use these coordinates as our primary key between the two meshes and renormalize our indices based on that.

Since our decimated mesh is a subset of the original, we can guarantee that every vertex in this mesh will exist in the original. By this logic, we shall renormalize our original mesh based on the vertices from the decimated one. Here is a visual that explains this.

The main workflow we'd like to implement is as follows:

1) load the raw obj file to pymeshlab data.
2) make note of the vertices and their raw indicies
3) decimate our mesh using the `optimalposition=False` parameter
4) create a merged dataframe joining based on the coordinate info
5) map the vertex indices based on pre-tranformation index --> post transformation index.
6) recreate the face dataframe based on these new vertices.

In theory, this process should maintain the shape of both meshes (original and decimated), while ensuring index consistency across their geometry. This should solve our problem statement in the markdown file [mesh-simplification.md](../mesh-simplification.md).

Eventually, this will be converted in to script format and called recursively to create these decimated meshes on demand.

Let's start by clearing data from our `MeshSet` object.

In [15]:
ms.clear()

We load our foot model once again.

In [16]:
ms.load_new_mesh('../../../models/foot/human-foot.obj')

Let's get our baseline vertices and faces, and convert those to dataframe.

In [17]:
m = ms.current_mesh()

v_matrix_bs = m.vertex_matrix()
f_matrix_bs = m.face_matrix()

baseline_vertex_df = pd.DataFrame(v_matrix_bs, columns = ["X", "Y", "Z"])
baseline_face_df = pd.DataFrame(f_matrix_bs, columns = ["v1", "v2", "v3"])

baseline_vertex_df.head()

,X,Y,Z
0,0.059608,0.383419,-0.047925
1,-0.030599,-0.016010,-0.020009
2,0.052395,0.402345,0.102574
3,0.049393,-0.015811,-0.022423
4,-0.061567,0.421855,-0.066727


In [18]:
baseline_face_df.head()

,v1,v2,v3
0,715,31,33
1,715,33,712
2,86,16,17
3,86,17,83
4,59,795,796


As a reminder, the `face` dataframe contains references to the index of the specific vertex that makes up its 3 corners.

Let's proceed with decimating our mesh, remembering to set the `optimalplacement` parameter equal to `False`.

In [19]:
ms.meshing_decimation_quadric_edge_collapse(optimalplacement=False)

Now, the vertices and faces corresponding to this mesh will be that of our decimated object. This means less numbers of vertices, and the indices will not align. We extract this information out and run some tests to verify that this is the case.

In [20]:
m = ms.current_mesh()

v_matrix_dm = m.vertex_matrix()
f_matrix_dm = m.face_matrix()

decimated_vertex_df = pd.DataFrame(v_matrix_dm, columns = ["X", "Y", "Z"])
decimated_face_df = pd.DataFrame(f_matrix_dm, columns = ["v1", "v2", "v3"])

len(decimated_vertex_df)

403

In [21]:
len(decimated_face_df)

792

We now apply our concatentation step from above.

In [22]:
decimated_vertex_df["coords"] = decimated_vertex_df[["X", "Y", "Z"]].apply(tuple, axis=1)
baseline_vertex_df["coords"] = baseline_vertex_df[["X", "Y", "Z"]].apply(tuple, axis=1)

In these 2 dataframes, our `coords` column serves as a unique key, given that no 2 vertices overlap with each other. In this case, it might be best to set this column as our index.

In [23]:
baseline_vertex_df["vertex_index"] = baseline_vertex_df.index
baseline_vertex_df = baseline_vertex_df.set_index("coords")

decimated_vertex_df["vertex_index"] = decimated_vertex_df.index
decimated_vertex_df = decimated_vertex_df.set_index("coords")

baseline_vertex_df.head()

,X,Y,Z,vertex_index
coords,,,,
"(0.059608, 0.383419, -0.047925)",0.059608,0.383419,-0.047925,0
"(-0.030599, -0.01601, -0.020009)",-0.030599,-0.016010,-0.020009,1
"(0.052395, 0.402345, 0.102574)",0.052395,0.402345,0.102574,2
"(0.049393, -0.015811, -0.022423)",0.049393,-0.015811,-0.022423,3
"(-0.061567, 0.421855, -0.066727)",-0.061567,0.421855,-0.066727,4


We now perform a merge based on the index of both our columns.

In [24]:
merged_df_vertex = pd.merge(baseline_vertex_df, decimated_vertex_df, left_index=True, right_index=True, how="outer")
merged_df_vertex.head()

,X_x,Y_x,Z_x,vertex_index_x,X_y,Y_y,Z_y,vertex_index_y
coords,,,,,,,,
"(-0.224832, 0.030038, 0.526462)",-0.224832,0.030038,0.526462,645,-0.224832,0.030038,0.526462,325.0
"(-0.223318, 0.034579, 0.441157)",-0.223318,0.034579,0.441157,98,-0.223318,0.034579,0.441157,92.0
"(-0.22168, 0.043812, 0.651764)",-0.221680,0.043812,0.651764,299,-0.221680,0.043812,0.651764,193.0
"(-0.221125, 0.076088, 0.533599)",-0.221125,0.076088,0.533599,644,-0.221125,0.076088,0.533599,324.0
"(-0.22075, 0.085796, 0.448143)",-0.220750,0.085796,0.448143,99,-0.220750,0.085796,0.448143,93.0


We now have a mapping of our vertices from the original mesh to the decimated one. The goal is to treat the decimated one as "ground truth" and extrapolate from there. Here, we first sort by our decimated mesh's index and reset the index of our overall dataframe.

In [25]:
merged_df_vertex = merged_df_vertex.sort_values(by="vertex_index_y")
merged_df_vertex = merged_df_vertex.reset_index()
merged_df_vertex

,coords,X_x,Y_x,Z_x,vertex_index_x,X_y,Y_y,Z_y,vertex_index_y
0,"(0.059608, 0.383419, -0.047925)",0.059608,0.383419,-0.047925,0,0.059608,0.383419,-0.047925,0.0
1,"(-0.030599, -0.01601, -0.020009)",-0.030599,-0.016010,-0.020009,1,-0.030599,-0.016010,-0.020009,1.0
2,"(0.052395, 0.402345, 0.102574)",0.052395,0.402345,0.102574,2,0.052395,0.402345,0.102574,2.0
3,"(0.049393, -0.015811, -0.022423)",0.049393,-0.015811,-0.022423,3,0.049393,-0.015811,-0.022423,3.0
4,"(-0.061567, 0.421855, -0.066727)",-0.061567,0.421855,-0.066727,4,-0.061567,0.421855,-0.066727,4.0
...,...,...,...,...,...,...,...,...,...
795,"(0.15269, 0.044007, 0.587606)",0.152690,0.044007,0.587606,775,NaN,NaN,NaN,NaN
796,"(0.154933, 0.074872, 0.516165)",0.154933,0.074872,0.516165,109,NaN,NaN,NaN,NaN
797,"(0.157481, 0.015595, 0.408668)",0.157481,0.015595,0.408668,795,NaN,NaN,NaN,NaN
798,"(0.163158, 0.035477, 0.397872)",0.163158,0.035477,0.397872,798,NaN,NaN,NaN,NaN


In [26]:
assert 235 == int(merged_df_vertex.loc[235]["vertex_index_y"])

This merged_df essentially serves as a map from our decimated mesh index space to our original mesh index space.

In [27]:
mapping_dict = {}

for i in merged_df_vertex.index:
    mapping_dict[ merged_df_vertex.loc[i]["vertex_index_x"] ] = i

mapping_dict

{np.int64(0): 0,
 np.int64(1): 1,
 np.int64(2): 2,
 np.int64(3): 3,
 np.int64(4): 4,
 np.int64(5): 5,
 np.int64(6): 6,
 np.int64(7): 7,
 np.int64(8): 8,
 np.int64(9): 9,
 np.int64(10): 10,
 np.int64(11): 11,
 np.int64(12): 12,
 np.int64(14): 13,
 np.int64(15): 14,
 np.int64(16): 15,
 np.int64(17): 16,
 np.int64(18): 17,
 np.int64(19): 18,
 np.int64(20): 19,
 np.int64(21): 20,
 np.int64(22): 21,
 np.int64(23): 22,
 np.int64(25): 23,
 np.int64(26): 24,
 np.int64(28): 25,
 np.int64(29): 26,
 np.int64(30): 27,
 np.int64(31): 28,
 np.int64(32): 29,
 np.int64(33): 30,
 np.int64(34): 31,
 np.int64(35): 32,
 np.int64(36): 33,
 np.int64(37): 34,
 np.int64(38): 35,
 np.int64(39): 36,
 np.int64(40): 37,
 np.int64(41): 38,
 np.int64(42): 39,
 np.int64(43): 40,
 np.int64(45): 41,
 np.int64(46): 42,
 np.int64(47): 43,
 np.int64(48): 44,
 np.int64(49): 45,
 np.int64(50): 46,
 np.int64(51): 47,
 np.int64(52): 48,
 np.int64(53): 49,
 np.int64(54): 50,
 np.int64(55): 51,
 np.int64(56): 52,
 np.int64(57)

Now we recreate our face dataframe from the original mesh using this transformed index space. As a reminder, here is what the original dataframe looks like.

In [28]:
baseline_face_df_reconstruct = baseline_face_df.copy()
baseline_face_df_reconstruct.head()

,v1,v2,v3
0,715,31,33
1,715,33,712
2,86,16,17
3,86,17,83
4,59,795,796


Now, we map these original values to the new ones using the `.map()` method and our created mapping dictionary.

In [29]:
baseline_face_df_reconstruct["v1_new"] = baseline_face_df_reconstruct["v1"].map(mapping_dict)
baseline_face_df_reconstruct.head()

,v1,v2,v3,v1_new
0,715,31,33,366
1,715,33,712,366
2,86,16,17,81
3,86,17,83,81
4,59,795,796,55


Let's confirm that the mapping went as expected.

In [30]:
assert baseline_face_df_reconstruct.loc[0]["v1_new"] == mapping_dict[baseline_face_df_reconstruct.loc[0]["v1"]]
assert baseline_face_df_reconstruct.loc[7]["v1_new"] == mapping_dict[baseline_face_df_reconstruct.loc[7]["v1"]]

Looks good, let's map the rest of the vertices as well.

In [31]:
baseline_face_df_reconstruct["v2_new"] = baseline_face_df_reconstruct["v2"].map(mapping_dict)
baseline_face_df_reconstruct["v3_new"] = baseline_face_df_reconstruct["v3"].map(mapping_dict)

baseline_face_df_reconstruct.head()

,v1,v2,v3,v1_new,v2_new,v3_new
0,715,31,33,366,28,30
1,715,33,712,366,30,363
2,86,16,17,81,15,16
3,86,17,83,81,16,78
4,59,795,796,55,797,401


Perfect. We have successfully remapped our original face dataframe.

## Reconversion to OBJ file

Now, its time to export these meshes out to new OBJ files that share the same geometry array structure. First, we need to extract the corresponding information out from our merged dataframes. Here is the data we need.

1) vertex array for original mesh: This data is saved in our `merged_df_vertex` in the columns "X_x", "Y_x", "Z_x". 
2) vertex array for decimated mesh: This data will be the original `v_matrix_dm` object we created after decimation.
3) face array for the original mesh: This data is saved in `baseline_face_df_reconstruct` in the columns "v1_new", "v2_new", "v3_new"
4) face array for the decimated mesh: This data is the original face array from our decimated mesh `f_matrix_dm`. As a reminder, this dataset correctly references our decimated mesh's indices.

We need to convert dataframes to numpy.

In [32]:
v_full = merged_df_vertex[["X_x", "Y_x", "Z_x"]].to_numpy()
v_dec = v_matrix_dm

f_full = baseline_face_df_reconstruct[["v1_new", "v2_new", "v3_new"]].to_numpy()
f_dec = f_matrix_dm

Creating a mesh involves passing the Vertex and Face array to a `Mesh()` class in pymeshlab.

In [33]:
# Create new meshes in pymeshlab by passing the arguments (vertex_array, faces_array)
m_orig = pymeshlab.Mesh(v_full, f_full)
m_dec = pymeshlab.Mesh(v_dec, f_dec)

Now, lets convert to obj file. First, we add the mesh to our Meshset() object.

In [34]:
ms.add_mesh(m_orig, "OriginalMesh")
ms.save_current_mesh("models" + "original_mesh.obj")

In [35]:
ms.add_mesh(m_dec, "DecimatedMesh")
ms.save_current_mesh("models" + "decimated_mesh.obj")

And what do these meshes look like? We load both files into Blender.

![Results of the Reconstructed OBJ file](../img/optimized-decimate-script-results.png)

Fantastic results. We see that the decimation has occured and removed the bulk of vertices in high concentration areas, specifically the toes. Now its time to see if our vertex array is truly shared between these meshes.

Loading the 2 meshes into the MeshLab application, this is what we observe.

![Vertex Indices match across indices in MeshLab](../img/same-vertex-index-after-decimation.png)

The vertex number 174 is common across both meshes.

Moreover, we load our two meshes as [LOD's to our BatchedMesh object in three.js](../../optimizing-the-scene/batchedmesh-with-LOD.md) scene. In this scene, our low resolution mesh is set as a lower resoultion LOD of the higher resolution one. The meshes are triggered to switch at a distance of 5 units from the camera. Here are the results.

The results were exactly what we'd hoped for, so its very good news that our method is now working.

Let's try other values for the mesh quality.

## Convert to Module

It is now time to save this as a module which we can call on other meshes. Let us recall the order of steps which needs to be conducted from the work above.

1) Load OBJ file to memory.
2) Get vertex and face arrays
3) Apply `ms.meshing_decimation_quadric_edge_collapse(optimalplacement=False)` algorithm on the mesh
4) Get decimated mesh's vertex and face arrays
5) Merge both arrays based on the coordinates of each vertex (should be UID for both)
6) Renormalize original mesh to decimated mesh's index structure

We shall create a basic structure of the function here, and then export it our to a module. 

As a reminder, here are the basic list of commands we sent to create our reconstrution.

In [36]:
# ms.load_new_mesh(obj_path)
# m = ms.current_mesh()

# v_matrix_bs = m.vertex_matrix()
# f_matrix_bs = m.face_matrix()

# baseline_vertex_df = pd.DataFrame(v_matrix_bs, columns = ["X", "Y", "Z"])
# baseline_face_df = pd.DataFrame(f_matrix_bs, columns = ["v1", "v2", "v3"])

# ms.meshing_decimation_quadric_edge_collapse(optimalplacement=False)

# m = ms.current_mesh()

# v_matrix_dm = m.vertex_matrix()
# f_matrix_dm = m.face_matrix()

# decimated_vertex_df = pd.DataFrame(v_matrix_dm, columns = ["X", "Y", "Z"])
# decimated_face_df = pd.DataFrame(f_matrix_dm, columns = ["v1", "v2", "v3"])

# decimated_vertex_df["coords"] = decimated_vertex_df[["X", "Y", "Z"]].apply(tuple, axis=1)
# baseline_vertex_df["coords"] = baseline_vertex_df[["X", "Y", "Z"]].apply(tuple, axis=1)

# baseline_vertex_df["vertex_index"] = baseline_vertex_df.index
# baseline_vertex_df = baseline_vertex_df.set_index("coords")

# decimated_vertex_df["vertex_index"] = decimated_vertex_df.index
# decimated_vertex_df = decimated_vertex_df.set_index("coords")

# merged_df_vertex = pd.merge(baseline_vertex_df, decimated_vertex_df, left_index=True, right_index=True, how="outer")

# merged_df_vertex = merged_df_vertex.sort_values(by="vertex_index_y")
# merged_df_vertex = merged_df_vertex.reset_index()

# mapping_dict = {}

# for i in merged_df_vertex.index:
#     mapping_dict[ merged_df_vertex.loc[i]["vertex_index_x"] ] = i

# baseline_face_df_reconstruct = baseline_face_df.copy()

# baseline_face_df_reconstruct["v1_new"] = baseline_face_df_reconstruct["v1"].map(mapping_dict)

# assert baseline_face_df_reconstruct.loc[0]["v1_new"] == mapping_dict[baseline_face_df_reconstruct.loc[0]["v1"]]
# assert baseline_face_df_reconstruct.loc[7]["v1_new"] == mapping_dict[baseline_face_df_reconstruct.loc[7]["v1"]]

# baseline_face_df_reconstruct["v2_new"] = baseline_face_df_reconstruct["v2"].map(mapping_dict)
# baseline_face_df_reconstruct["v3_new"] = baseline_face_df_reconstruct["v3"].map(mapping_dict)

# v_full = merged_df_vertex[["X_x", "Y_x", "Z_x"]].to_numpy()
# v_dec = v_matrix_dm

# f_full = baseline_face_df_reconstruct[["v1_new", "v2_new", "v3_new"]].to_numpy()
# f_dec = f_matrix_dm

# m_orig = pymeshlab.Mesh(v_full, f_full)
# m_dec = pymeshlab.Mesh(v_dec, f_dec)

# ms.add_mesh(m_orig, "OriginalMesh")
# ms.save_current_mesh("models" + "original_mesh.obj")

# ms.add_mesh(m_dec, "DecimatedMesh")
# ms.save_current_mesh("models" + "decimated_mesh.obj")


# # print(dict(enumerate(v_dm)))

# #     combined_v = np.vstack((v_dm, v_org))

# #     unique_v, first_idx, inv_indices = np.unique(
# #         combined_v, axis=0, return_index=True, return_inverse=True, sorted=False
# #     )

#     # sort_order = np.argsort(first_idx)
#     # final_v_buffer = unique_v[sort_order]
    
#     # # 3. Create a re-mapping for the inversion
#     # # This aligns the inverse indices with the sorted unique buffer
#     # reindex_map = np.zeros_like(sort_order)
#     # reindex_map[sort_order] = np.arange(len(sort_order))
#     # final_mapping = reindex_map[inv_indices]

#     # # 4. Extract Face Matrices
#     # # f_dec: Since v_dm was at the front, its indices map to 0 -> len(v_dm)-1
#     # # Because we used return_index and sorted, the decimated indices remain 1:1
#     # f_dec_final = final_mapping[:len(v_dm)][np.arange(len(v_dm))][f_dm] 
    
#     # # f_orig: The original mesh now points to the combined buffer
#     # f_orig_mapping = final_mapping[len(v_dm):]
#     # f_orig_final = f_orig_mapping[f_org]

#     # m_orig = pymeshlab.Mesh(v_full, f_full)
#     # m_dec = pymeshlab.Mesh(v_dec, f_dec)

#     # ms.add_mesh(m_orig, "OriginalMesh")
#     # ms.save_current_mesh("models" + "original_mesh.obj")

#     # ms.add_mesh(m_dec, "DecimatedMesh")
#     # ms.save_current_mesh("models" + "decimated_mesh.obj")

I'd like to recreate this without pandas. Using only numpy, this problem becomes slightly more complex, but manageable. The basic structure of our function is as follows -->

- Get current object's vertex and face array (`v_org` and `f_org`)
- Apply decimation script with `optimalplacement=False`
- Get decimated mesh's vertex and face array (`v_dm` and `f_dm`)
- Create mapping dictionary- for each vertex in `v_dm`, save the coordinates as the key, and index as the map
- Iterate through every vertex in `v_org`, looking up the coordinates of each one in our mapping dictionary. If it exists, replace the element with the value from the map. If not, set to infinity.
- Sort this new array ascending. This means the vertex order will mimic that of the decimated mesh, while the infinity values will get appended to the end.
- Remap the original vertex and face arrays with this new order.

In [5]:
def decimate_mesh(obj_path, perc_red=0.0):
    v_dict = {}
    
    ms.load_new_mesh(obj_path)
    m = ms.current_mesh()
    
    v_org = m.vertex_matrix()
    f_org = m.face_matrix()

    ms.meshing_decimation_quadric_edge_collapse(targetperc=perc_red, optimalplacement=False)
    m = ms.current_mesh()

    v_dm = m.vertex_matrix()
    f_dm = m.face_matrix()

    v_dict = { tuple(row): i for i, row in enumerate(v_dm) }
    v_remapping = np.argsort(np.array([v_dict.get(tuple(row), np.inf) for row in v_org ]))

    v_org_rmp = v_org[v_remapping]
    f_org_rmp = v_remapping[f_org]

    assert v_org_rmp[:len(v_dm)].all() == v_dm.all()

    ms.clear()

    return v_org_rmp, f_org_rmp, v_dm, f_dm


In [38]:
v_recon, f_recon, v_dm, f_dm = decimate_mesh('../../../models/foot/human-foot.obj')

We include the following assert statement to ensure the function is working as intended.

In [39]:
assert v_recon[:len(v_dm)].all() == v_dm.all()

## Testing With Additional Meshes

Let's run some tests on this script with additional objects. I'm curious how this would work, especially with non manifold geometry. We shall be applying this function on the following 3 objects.

![Additional Testing on these objects](../img/decimation-additional-testing.png)

Object 1 is a valve handle. This object has intricately modelled geometry, and I expect a good decimated output from it.
Object 2 is a generic steel structure. I included this object since it already has a low count of vertices within it, and I'm curious to see how the decimation algorithm approaches it.
Object 3 is a generic pipe. This object is specially chosen since it has non manifold geometry- i.e. an open end. This mesh theoretically cannot exist in nature since on one side, we have only an open edge loop.

![Non Manifold Geometry Example](../img/non-manifold-geometry.png)

We observe in the figure above, one end of the pipe is simply a line segment. This will be a problem for applications like 3D printing, but pipes like these are littered around 3D models. Our function will need to handle this.

We start first, with our valve.

In [6]:
ms.clear()

v_recon_valve, f_recon_valve, v_dm_valve, f_dm_valve = decimate_mesh('../../../models/piperack/piperacks_valve only.obj')

Let's create a new set of meshes from this object and see what they look like.

In [7]:
valve_orig = pymeshlab.Mesh(v_recon_valve, f_recon_valve)
valve_dec = pymeshlab.Mesh(v_dm_valve, f_dm_valve)

ms.add_mesh(valve_orig, "OriginalMesh")
ms.save_current_mesh("valve_reconstruct.obj")

ms.add_mesh(valve_dec, "DecimatedMesh")
ms.save_current_mesh("valve_decimate.obj")

Loading them to Blender and MeshLab, this is what we see.

![Preliminary Results for Valve Decimation](../img/valve-decimation-first-results.png)

Not good. The object in the center is our decimated mesh, the object on the right is our original. We can see that the decimation occured properly- a visual indicator is the smoothness of the bolt in the middle of the valve.

However, the object on the right is the reconstructed version of the original, which looks completely wrong. It seems like the vertex indices have been messed up, we see the familiar "exploded" look that we saw when working initially with the [BatchedMeshLOD object](../../optimizing-the-scene/batchedmesh-with-LOD.md).

Let's inspect.

In [48]:
v_recon_valve

array([[-3.00117 ,  0.653368, -0.941665],
       [-3.001998,  0.674464, -0.941665],
       [-3.001998,  0.674464, -0.915417],
       ...,
       [-3.022813,  0.651252, -0.909043],
       [-3.011684,  0.672403, -0.909043],
       [-3.016895,  0.675144, -0.909043]], shape=(3606, 3))

In [47]:
v_dm_valve

array([[-3.00117 ,  0.653368, -0.941665],
       [-3.001998,  0.674464, -0.941665],
       [-3.001998,  0.674464, -0.915417],
       ...,
       [-3.057161,  0.451321, -0.804565],
       [-3.061818,  0.461098, -0.812225],
       [-3.062558,  0.454849, -0.812225]], shape=(1834, 3))

In [49]:
assert v_recon_valve[:len(v_dm_valve)].all() == v_dm_valve.all()

Seems like the vertex structure is the same across the meshes. What about the faces?

In [50]:
f_dm_valve

array([[238, 239, 236],
       [236, 237, 238],
       [241, 238, 237],
       ...,
       [231, 221, 224],
       [224, 232, 231],
       [226, 235, 234]], shape=(3544, 3), dtype=int32)

In [51]:
f_recon_valve

array([[362, 363, 360],
       [360, 361, 362],
       [365, 362, 361],
       ...,
       [340, 356, 355],
       [357, 339, 342],
       [342, 358, 357]], shape=(7088, 3))

Its hard to tell where the face matrix might be differing. For now, let's apply this filter to more meshes and see if its a one-off or consistently being done across the board.

Here, we apply the decimation filter to our steel structure.

In [52]:
ms.clear()

v_recon_steel, f_recon_steel, v_dm_steel, f_dm_steel = decimate_mesh('../../../models/piperack/piperacks_steel_structure.obj')

In [53]:
steel_orig = pymeshlab.Mesh(v_recon_steel, f_recon_steel)
steel_dec = pymeshlab.Mesh(v_dm_steel, f_dm_steel)

ms.add_mesh(steel_orig, "OriginalMesh")
ms.save_current_mesh("steel_reconstruct.obj")

ms.add_mesh(steel_dec, "DecimatedMesh")
ms.save_current_mesh("steel_decimate.obj")

![Steel Structure Preliminary results](../img/steel-decimate-preliminary-results.png)

Seems to be happening here too.

In [54]:
ms.clear()

v_recon_foot_2, f_recon_foot_2, v_dm_foot_2, f_dm_foot_2 = decimate_mesh('../../../models/foot/human-foot.obj')

In [55]:
foot_2_orig = pymeshlab.Mesh(v_recon_foot_2, f_recon_foot_2)
foot_2_dec = pymeshlab.Mesh(v_dm_foot_2, f_dm_foot_2)

ms.add_mesh(foot_2_orig, "OriginalMesh")
ms.save_current_mesh("foot_2_reconstruct.obj")

ms.add_mesh(foot_2_dec, "DecimatedMesh")
ms.save_current_mesh("foot_2_decimate.obj")

This looks successful, so we proceed with converting this to a module. The raw code for this function is saved in the `scripts` folder of this `reducing-mesh-density` subfolder.